In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import csv
import numpy as np
import copy

carico il subset dei commenti che ho taggato violenti e li analizzo.

prima divido per tossicità (3 livelli), poi applico lo split dei dati e infine i modelli di regressione

In [3]:
df = pd.read_csv(
    "../datasets/df_violence.csv",
    sep=",",
    quoting=csv.QUOTE_MINIMAL,
    encoding="utf-8",
    engine="python"
)
df

,commento,valutazione,tossicita,tipo_tossicita,age_group,gender,utente_id,emoji_decoded,emoji_list,emoji_count,...,Top20_kill,Top20_like,Top20_make,Top20_mother,Top20_murder,Top20_people,Top20_rape,Top20_shoot,Top20_want,Top20_would
0,Think carefully. Accuse me of raping kids and ...,Extremely violent,3,4,1,1,1,think carefully accuse raping kids going rape ...,[],0,...,0,0,0,0,0,0,0,0,0,0
1,Maybe someone should choke you and your immedi...,Extremely violent,3,4,1,1,1,maybe someone choke immediate family death see...,[],0,...,0,0,0,0,0,0,0,0,0,0
2,I will murder your family friend!,Extremely violent,3,4,1,1,1,murder family friend,[],0,...,0,0,0,0,0,0,0,0,0,0
3,"Eww, refer to me as a disgusting subhuman Celt...",Extremely violent,3,4,1,1,1,eww refer disgusting subhuman celt blood eagle...,[],0,...,0,0,0,0,0,0,0,0,0,0
4,And then he rapes you,Extremely violent,3,4,1,1,1,rapes,[],0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1062,Commies are going to commie. These people hate...,Extremely violent,3,4,1,0,211,commies going commie people hate want die,[],0,...,0,0,0,0,0,0,0,0,0,0
1063,Yeah but then the good guys win and rape you t...,Extremely violent,3,4,1,0,211,yeah good guys win rape death occupy nation in...,[],0,...,0,0,0,0,0,0,0,0,0,0
1064,If you're white and use your gun to defend you...,A little violent,1,4,1,0,211,white use gun defend arrested racist get life ...,[],0,...,0,0,0,0,0,0,0,0,0,0
1065,I will rape Elon Musk,A little violent,1,4,1,0,211,rape elon musk,[],0,...,0,0,0,0,0,0,0,0,0,0


### PREPROCESSING

controllo dati mancanti e altre anomalie

In [4]:
df.isnull().sum()

commento          0
valutazione       0
tossicita         0
tipo_tossicita    0
age_group         0
                 ..
Top20_people      0
Top20_rape        0
Top20_shoot       0
Top20_want        0
Top20_would       0
Length: 66, dtype: int64

In [5]:
df["badwords_matches"].fillna("", inplace=True)

In [6]:
df.isnull().sum()

commento          0
valutazione       0
tossicita         0
tipo_tossicita    0
age_group         0
                 ..
Top20_people      0
Top20_rape        0
Top20_shoot       0
Top20_want        0
Top20_would       0
Length: 66, dtype: int64

In [7]:
doubles = df.duplicated()
doubles.sum() # non sono veri duplicati perché utente è differente, quindi niente duplicati

2

### DIVISIONE DEI DATI

In [8]:
# Funzione per generare una colonna isToxic in base ai livelli di sensibilità (3)
def imposta_isToxic(df, sensibilita):
    if sensibilita == "poco":
        df["isToxic"] = df["tossicita"].apply(lambda x: 1 if x == 3 else 0)
    elif sensibilita == "media":
        df["isToxic"] = df["tossicita"].apply(lambda x: 1 if x >= 2 else 0)
    elif sensibilita == "alta":
        df["isToxic"] = df["tossicita"].apply(lambda x: 1 if x >= 1 else 0)
    return df

# Crea 3 versioni del dataset
df_bassa_sens = imposta_isToxic(df.copy(), sensibilita="poco")
df_media_sens = imposta_isToxic(df.copy(), sensibilita="media")
df_alta_sens = imposta_isToxic(df.copy(), sensibilita="alta")

In [9]:
df_bassa_sens[["tossicita", "isToxic"]].groupby("isToxic").count()

,tossicita
isToxic,
0,420
1,647


In [10]:
df_media_sens[["tossicita", "isToxic"]].groupby("isToxic").count()

,tossicita
isToxic,
0,134
1,933


In [12]:
df_alta_sens[["tossicita", "isToxic"]].groupby("isToxic").count()

,tossicita
isToxic,
0,25
1,1042


### 01. MODELLO VIOLENZA - BASSA SENSIBILITÀ

In [13]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

In [16]:
# 2. Crea pipeline: TF-IDF + Classificatore
pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(lowercase=True, stop_words='english', ngram_range=(1,2))),
    ("clf", LogisticRegression(max_iter=1000))
])

train, val = train_test_split(df_bassa_sens, test_size=0.7, stratify=df_bassa_sens["isToxic"], random_state=42)
pipeline.fit(train["emoji_decoded"], train["isToxic"])
print("Validation accuracy:", pipeline.score(val["emoji_decoded"], val["isToxic"]))

Validation accuracy: 0.6372155287817939


### MODELLO VIOLENZA - MEDIO SENSIBILITÀ

### MODELLO VIOLENZA - ALTA SENSIBILITÀ